# Gold Layer — Drug Delivery Dataset (v2 — Refined Star Schema)

---

**Pipeline stage:** `Silver -> Gold`

**Objective:** Transform the cleaned Silver dataset into a dimensional model (Star Schema) made of **one Fact table and seven Dimension tables**, ready for Power BI, Tableau, or any SQL-based Data Warehouse.

**Input:** `DrugDelivery_Silver.csv` (Silver layer output, 2374 rows x 25 columns)

**Output:** `Dim_Nanoparticle.csv`, `Dim_Tumor.csv`, `Dim_Time.csv`, `Dim_Targeting_Strategy.csv`, `Dim_Organ.csv`, `Dim_Administration.csv`, `Dim_Efficiency.csv`, `Fact_Biodistribution.csv`

**Environment:** Google Colab

---

### What changed vs. the first version

The first version of this Gold Layer grouped `Administration Dosages (mg/kg)`, `Organ or tissue`, and `Targeting Strategy` into a single `Dim_Administration` table (368 unique combinations). That works, but it has two practical costs in a BI tool:

1. **`Targeting Strategy`** only has **2** distinct values (`Active` / `Passive`). Slicing on it means scanning a 368-row table to find 2 values.
2. **`Organ or tissue`** only has **17** distinct values. Same problem — it is a true descriptive attribute, not something that needs to ride along with a near-continuous dosage number.

So in this version, `Dim_Administration` is split into **three independent pieces**:

| Old design | New design |
|---|---|
| `Dim_Administration` (Dosage + Organ + Targeting Strategy, 368 rows) | `Dim_Targeting_Strategy` (2 rows) |
| | `Dim_Organ` (17 rows) |
| | `Dim_Administration` (Dosage only, 42 rows) |

This keeps every dimension table representing **one single logical concept** (a core Star Schema modelling principle), makes filters on Targeting Strategy / Organ instant, and keeps the dosage dimension small and clean.

### What this notebook does

| # | Step | Technique |
|---|------|-----------|
| 1 | Load the Silver dataset | pandas |
| 2 | Define the Business Process (what we are measuring) | Conceptual design |
| 3 | Build `Dim_Nanoparticle` | Drop duplicates + Surrogate Key |
| 4 | Build `Dim_Tumor` | Drop duplicates + Surrogate Key |
| 5 | Build `Dim_Time` | Drop duplicates + Surrogate Key |
| 6 | Build `Dim_Targeting_Strategy` | Drop duplicates + Surrogate Key |
| 7 | Build `Dim_Organ` | Drop duplicates + Surrogate Key |
| 8 | Build `Dim_Administration` | Drop duplicates + Surrogate Key |
| 9 | Build `Dim_Efficiency` | Ordinal mapping + Surrogate Key |
| 10 | Build `Fact_Biodistribution` | Merge keys + select measures |
| 11 | Validate the Star Schema | Row counts + null checks |
| 12 | Export the Gold tables | CSV export |




## 1. Setup & Imports

### 1.1 Mount Google Drive
Run this cell only if the Silver CSV is stored on Google Drive.

In [ ]:
# Mount Google Drive (skip this cell if the file is uploaded locally)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### 1.2 Import libraries
We only need `pandas` for the transformations in this notebook. `numpy` is imported as well since it is commonly needed for numeric checks.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np

# Display settings (optional, makes wide tables easier to read)
pd.set_option('display.max_columns', None)



## 2. Load the Silver Layer Dataset

In [ ]:
# Path to the Silver CSV file
SILVER_PATH = '/content/drive/MyDrive/Medallion_Arch/DrugDelivery_Silver.csv'  # update path if needed

df = pd.read_csv(SILVER_PATH)

print('Rows:', df.shape[0])
print('Columns:', df.shape[1])
df.head()

Rows: 2374
Columns: 26


,NPs ID,NP_Class,INPs_Core,Targeting Strategy,Shape,Size (nm),Size_Category,Zeta Potential (mv),Zeta_Category,Responsive release,Tumor Cell,Tumor Site,Tumor Size ( mm³),Administration Dosages (mg/kg),Organ or tissue,Time point (h),Biodistribution (%ID),Tumor_%ID,Liver_%ID,Selectivity_Index,Targeting_Efficiency,Shell Type,MW Shell,HAS_PEG,Spleen_%ID,Time_Phase
0,INPs-P1,Inorganic,Gold,Active,Rod,37.5,Optimal Range,-18.0,Negative,No,HeLa cell,Cervix,100.0,20.0,Tumor,6.0,1.6400,1.64,29.03,0.051589,Moderate,PEG,4441,Yes,2.76,Early
1,INPs-P1,Inorganic,Gold,Active,Rod,37.5,Optimal Range,-18.0,Negative,No,HeLa cell,Cervix,100.0,20.0,Tumor,24.0,2.0600,2.06,31.70,0.059281,Moderate,PEG,4441,Yes,3.05,Early
2,INPs-P1,Inorganic,Gold,Active,Rod,37.5,Optimal Range,-18.0,Negative,No,HeLa cell,Cervix,100.0,20.0,Tumor,168.0,2.0100,2.01,21.74,0.083024,Moderate,PEG,4441,Yes,2.47,Late
3,INPs-P1,Inorganic,Gold,Active,Rod,37.5,Optimal Range,-18.0,Negative,No,HeLa cell,Cervix,100.0,20.0,Heart,6.0,0.4174,1.64,29.03,0.051589,Moderate,PEG,4441,Yes,2.76,Early
4,INPs-P1,Inorganic,Gold,Active,Rod,37.5,Optimal Range,-18.0,Negative,No,HeLa cell,Cervix,100.0,20.0,Heart,24.0,0.3728,2.06,31.70,0.059281,Moderate,PEG,4441,Yes,3.05,Early


In [ ]:
# Quick look at column names and data types
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2374 entries, 0 to 2373
Data columns (total 26 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   NPs ID                          2374 non-null   object 
 1   NP_Class                        2374 non-null   object 
 2   INPs_Core                       2374 non-null   object 
 3   Targeting Strategy              2374 non-null   object 
 4   Shape                           2374 non-null   object 
 5   Size (nm)                       2374 non-null   float64
 6   Size_Category                   2374 non-null   object 
 7   Zeta Potential (mv)             1763 non-null   float64
 8   Zeta_Category                   1763 non-null   object 
 9   Responsive release              2374 non-null   object 
 10  Tumor Cell                      2374 non-null   object 
 11  Tumor Site                      2374 non-null   object 
 12  Tumor Size ( mm³)               23



## 3. Star Schema Design

### 3.1 Business Process

Before building any table, we ask: **what are we measuring?**

In this dataset, every row is one observation of how a nanoparticle behaves inside a tumor model: how much of it reaches the tumor, the liver, and the spleen, and how selective and efficient that targeting is.

So the Business Process is: **Nanoparticle Biodistribution Measurement**.

### 3.2 Fact Table

`Fact_Biodistribution` holds the numeric measures (the things we calculate and aggregate) plus the foreign keys that connect it to the seven dimensions:

| Column | Type | Description |
|---|---|---|
| Fact_ID | Surrogate Key (PK) | Unique row identifier |
| NP_Key | Foreign Key | Link to Dim_Nanoparticle |
| Tumor_Key | Foreign Key | Link to Dim_Tumor |
| Time_Key | Foreign Key | Link to Dim_Time |
| Targeting_Key | Foreign Key | Link to Dim_Targeting_Strategy |
| Organ_Key | Foreign Key | Link to Dim_Organ |
| Administration_Key | Foreign Key | Link to Dim_Administration |
| Efficiency_Key | Foreign Key | Link to Dim_Efficiency |
| Biodistribution (%ID) | Measure | Overall % of injected dose found |
| Tumor_%ID | Measure | % of injected dose reaching the tumor |
| Liver_%ID | Measure | % of injected dose reaching the liver |
| Spleen_%ID | Measure | % of injected dose reaching the spleen |
| Selectivity_Index | Measure | Tumor targeting selectivity score |

### 3.3 Dimension Tables

**Dim_Nanoparticle** — describes the nanoparticle itself (its physical and chemical identity). Built from: `NPs ID`, `NP_Class`, `INPs_Core`, `Shape`, `Size (nm)`, `Size_Category`, `Zeta Potential (mv)`, `Zeta_Category`, `Responsive release`, `Shell Type`, `MW Shell`, `HAS_PEG`. **84 unique combinations.**

**Dim_Tumor** — describes the tumor model used in the experiment. Built from: `Tumor Cell`, `Tumor Site`, `Tumor Size ( mm³)`. **45 unique combinations.**

**Dim_Time** — describes the observation time point. Built from: `Time point (h)`, plus a derived `Time_Phase` attribute (Early 0–24h / Intermediate 24–96h / Late >96h) that groups the 29 raw time points into 3 readable phases for easier slicing in Power BI. **29 unique time points.**

**Dim_Targeting_Strategy** *(new)* — describes whether the nanoparticle used active or passive targeting. Built from: `Targeting Strategy`. **2 unique values** (`Active`, `Passive`).

**Dim_Organ** *(new)* — describes the organ or tissue where biodistribution was measured. Built from: `Organ or tissue`. **17 unique values.**

**Dim_Administration** *(refined)* — describes the dose given per kilogram of body weight only. Built from: `Administration Dosages (mg/kg)`. **42 unique dosages.**

**Dim_Efficiency** *(new)* — `Targeting_Efficiency` was originally treated as a fact measure, but it is really a descriptive, ordinal rating of how well the nanoparticle targeted the tumor, not a number we sum/average like the %ID measures. So it becomes its own dimension, with a fixed ordinal Surrogate Key (`Efficiency_Key`) so it sorts correctly (Very Poor -> Exceptional) in any BI tool instead of sorting alphabetically. **6 unique values:**

| Efficiency_Key | Targeting_Efficiency |
|---|---|
| 1 | Very Poor |
| 2 | Poor |
| 3 | Moderate |
| 4 | Moderately High |
| 5 | High |
| 6 | Exceptional |

> **Why split them?** `Targeting Strategy` and `Organ or tissue` are independent descriptive attributes — they do not change just because the dosage changes. Keeping them in one table with the dosage (as in the v1 design) forced every dosage/organ/strategy combination to be re-created as a new row (368 rows), even though there are really only 2 + 17 + 42 = 61 underlying distinct values. Splitting them follows the Star Schema principle that **each dimension should represent a single logical concept**, and makes slicing by Targeting Strategy or Organ much faster in Power BI / Tableau.

### 3.4 Star Schema Diagram

```
                              Dim_Tumor
                                  |
                Dim_Targeting_Strategy
                                  |
Dim_Time ------- Fact_Biodistribution ------- Dim_Nanoparticle
                                  |
                       Dim_Organ      Dim_Administration      Dim_Efficiency
```

### 3.5 Keys summary

- **PK (Primary Key):** uniquely identifies a row inside its own table (e.g. `NP_Key` inside `Dim_Nanoparticle`).
- **FK (Foreign Key):** a key inside the Fact table that points to the PK of a Dimension table (e.g. `NP_Key` inside `Fact_Biodistribution`).
- **SK (Surrogate Key):** an artificial, auto-generated integer key (1, 2, 3, ...) created by us instead of using the original business key. Surrogate keys make joins faster and are the standard practice in Data Warehouse design.



## 4. Build `Dim_Nanoparticle`

We select the nanoparticle descriptive columns, drop duplicate combinations, and add a Surrogate Key (`NP_Key`).

In [ ]:
# Columns that describe the nanoparticle
np_columns = [
    'NPs ID',
    'NP_Class',
    'INPs_Core',
    'Shape',
    'Size (nm)',
    'Size_Category',
    'Zeta Potential (mv)',
    'Zeta_Category',
    'Responsive release',
    'Shell Type',
    'MW Shell',
    'HAS_PEG'
]

# Drop duplicate rows to keep only unique nanoparticle records
dim_nanoparticle = df[np_columns].drop_duplicates().reset_index(drop=True)

# Add the surrogate primary key (NP_Key)
dim_nanoparticle.insert(0, 'NP_Key', range(1, len(dim_nanoparticle) + 1))

print('Dim_Nanoparticle shape:', dim_nanoparticle.shape)
dim_nanoparticle.head()

Dim_Nanoparticle shape: (84, 13)


,NP_Key,NPs ID,NP_Class,INPs_Core,Shape,Size (nm),Size_Category,Zeta Potential (mv),Zeta_Category,Responsive release,Shell Type,MW Shell,HAS_PEG
0,1,INPs-P1,Inorganic,Gold,Rod,37.5,Optimal Range,-18.0,Negative,No,PEG,4441,Yes
1,2,INPs-P2,Inorganic,Gold,Rod,44.7,Optimal Range,0.0,Near Neutral,No,PEG,Unknown,Yes
2,3,INPs-P3,Inorganic,Gold,Spherical,38.3,Optimal Range,-5.0,Near Neutral,No,PEG,5000,Yes
3,4,INPs-P4,Inorganic,Gold,Spherical,27.3,Optimal Range,NaN,NaN,No,No Stealth Effect,Not Exist,No
4,5,INPs-P5,Inorganic,Gold,Spherical,17.1,Optimal Range,-9.8,Near Neutral,No,No Stealth Effect,Not Exist,No




## 5. Build `Dim_Tumor`

Same logic: unique tumor combinations, each with its own Surrogate Key (`Tumor_Key`).

In [ ]:
# Columns that describe the tumor model
tumor_columns = [
    'Tumor Cell',
    'Tumor Site',
    'Tumor Size ( mm\u00b3)'
]

dim_tumor = df[tumor_columns].drop_duplicates().reset_index(drop=True)

# Add the surrogate primary key (Tumor_Key)
dim_tumor.insert(0, 'Tumor_Key', range(1, len(dim_tumor) + 1))

print('Dim_Tumor shape:', dim_tumor.shape)
dim_tumor.head()

Dim_Tumor shape: (45, 4)


,Tumor_Key,Tumor Cell,Tumor Site,Tumor Size ( mm³)
0,1,HeLa cell,Cervix,100.0
1,2,CT26.WT cell,Colon,100.0
2,3,9L.E29 cell,Brain,100.0
3,4,DU145 cell,Prostate,300.0
4,5,KB cell,Cervix,100.0




## 6. Build `Dim_Time`

Each unique time point gets one row, sorted in ascending order, with a Surrogate Key (`Time_Key`).

In [ ]:
# Column that describes the observation time point
dim_time = (
    df[['Time point (h)', 'Time_Phase']]
    .drop_duplicates()
    .sort_values('Time point (h)')
    .reset_index(drop=True)
)

# Add the surrogate primary key (Time_Key)
dim_time.insert(0, 'Time_Key', range(1, len(dim_time) + 1))

print('Dim_Time shape:', dim_time.shape)
dim_time.head(29)

Dim_Time shape: (29, 3)


,Time_Key,Time point (h),Time_Phase
0,1,0.000,Early
1,2,0.050,Early
2,3,0.083,Early
3,4,0.167,Early
4,5,0.250,Early
5,6,0.330,Early
6,7,0.500,Early
7,8,0.667,Early
8,9,1.000,Early
9,10,1.500,Early




## 7. Build `Dim_Targeting_Strategy`

`Targeting Strategy` only has 2 distinct values (`Active`, `Passive`). Pulling it out of `Dim_Administration` into its own tiny dimension makes it a one-click slicer in Power BI / Tableau, instead of a filter buried inside a 368-row table.

In [ ]:
# Column that describes the targeting strategy used
dim_targeting_strategy = (
    df[['Targeting Strategy']]
    .drop_duplicates()
    .sort_values('Targeting Strategy')
    .reset_index(drop=True)
)

# Add the surrogate primary key (Targeting_Key)
dim_targeting_strategy.insert(0, 'Targeting_Key', range(1, len(dim_targeting_strategy) + 1))

print('Dim_Targeting_Strategy shape:', dim_targeting_strategy.shape)
dim_targeting_strategy

Dim_Targeting_Strategy shape: (2, 2)


,Targeting_Key,Targeting Strategy
0,1,Active
1,2,Passive




## 8. Build `Dim_Organ`

`Organ or tissue` has 17 distinct values describing where biodistribution was measured (Tumor, Liver, Spleen, Lungs, Kidneys, Brain, ...). It is a standalone descriptive attribute, so it becomes its own dimension with a Surrogate Key (`Organ_Key`).

In [ ]:
# Column that describes the organ/tissue measured
dim_organ = (
    df[['Organ or tissue']]
    .drop_duplicates()
    .sort_values('Organ or tissue')
    .reset_index(drop=True)
)

# Add the surrogate primary key (Organ_Key)
dim_organ.insert(0, 'Organ_Key', range(1, len(dim_organ) + 1))

print('Dim_Organ shape:', dim_organ.shape)
dim_organ

Dim_Organ shape: (17, 2)


,Organ_Key,Organ or tissue
0,1,Blood
1,2,Bone
2,3,Brain
3,4,Heart
4,5,Intestine
5,6,Kidneys
6,7,Liver
7,8,Lungs
8,9,Muscle
9,10,Pancreas




## 9. Build `Dim_Administration`

After moving `Targeting Strategy` and `Organ or tissue` into their own dimensions, `Dim_Administration` now holds only the dosage values (`Administration Dosages (mg/kg)`), with a Surrogate Key (`Administration_Key`). This shrinks it from 368 rows down to 42 — one row per distinct dosage.

In [ ]:
# Column that describes the dose given per kilogram of body weight
admin_columns = ['Administration Dosages (mg/kg)']

dim_administration = (
    df[admin_columns]
    .drop_duplicates()
    .sort_values('Administration Dosages (mg/kg)')
    .reset_index(drop=True)
)

# Add the surrogate primary key (Administration_Key)
dim_administration.insert(0, 'Administration_Key', range(1, len(dim_administration) + 1))

print('Dim_Administration shape:', dim_administration.shape)
dim_administration.head()

Dim_Administration shape: (42, 2)


,Administration_Key,Administration Dosages (mg/kg)
0,1,1.350000e-09
1,2,1.500000e-01
2,3,2.500000e-01
3,4,3.000000e-01
4,5,1.000000e+00




## 10. Build `Dim_Efficiency`

`Targeting_Efficiency` was a Measure in the previous version, but it is actually a categorical, ordinal rating (`Very Poor` ... `Exceptional`), not something we sum or average. So it moves out of the Fact table and becomes its own dimension.

Unlike the other new dimensions (which were sorted alphabetically), here we map each label to a **fixed ordinal Surrogate Key** ourselves, so the ranking order (worst -> best) is preserved instead of being sorted alphabetically (which would put `Exceptional` before `High`).

In [ ]:
# Fixed ordinal mapping for Targeting Efficiency (worst -> best)
efficiency_order = {
    'Very Poor': 1,
    'Poor': 2,
    'Moderate': 3,
    'Moderately High': 4,
    'High': 5,
    'Exceptional': 6
}

dim_efficiency = pd.DataFrame({
    'Efficiency_Key': list(efficiency_order.values()),
    'Targeting_Efficiency': list(efficiency_order.keys())
})

print('Dim_Efficiency shape:', dim_efficiency.shape)
dim_efficiency

Dim_Efficiency shape: (6, 2)


,Efficiency_Key,Targeting_Efficiency
0,1,Very Poor
1,2,Poor
2,3,Moderate
3,4,Moderately High
4,5,High
5,6,Exceptional




## 11. Build `Fact_Biodistribution`

### 11.1 Merge Surrogate Keys back into the base table

We join each of the seven dimensions back onto the original Silver dataframe, using the same descriptive columns used to build that dimension. This brings every Surrogate Key into every row of the original data.

In [ ]:
# Start from a copy of the original Silver dataframe
fact_base = df.copy()

# Merge Dim_Nanoparticle keys
fact_base = fact_base.merge(dim_nanoparticle, on=np_columns, how='left')

# Merge Dim_Tumor keys
fact_base = fact_base.merge(dim_tumor, on=tumor_columns, how='left')

# Merge Dim_Time keys
fact_base = fact_base.merge(dim_time, on=['Time point (h)'], how='left')

# Merge Dim_Targeting_Strategy keys
fact_base = fact_base.merge(dim_targeting_strategy, on=['Targeting Strategy'], how='left')

# Merge Dim_Organ keys
fact_base = fact_base.merge(dim_organ, on=['Organ or tissue'], how='left')

# Merge Dim_Administration keys
fact_base = fact_base.merge(dim_administration, on=admin_columns, how='left')

# Merge Dim_Efficiency keys
fact_base = fact_base.merge(dim_efficiency, on=['Targeting_Efficiency'], how='left')

print('fact_base shape after merging all keys:', fact_base.shape)
fact_base[['NP_Key', 'Tumor_Key', 'Time_Key', 'Targeting_Key', 'Organ_Key', 'Administration_Key', 'Efficiency_Key']].head()


fact_base shape after merging all keys: (2374, 34)


,NPs ID,NP_Class,INPs_Core,Targeting Strategy,Shape,Size (nm),Size_Category,Zeta Potential (mv),Zeta_Category,Responsive release,Tumor Cell,Tumor Site,Tumor Size ( mm³),Administration Dosages (mg/kg),Organ or tissue,Time point (h),Biodistribution (%ID),Tumor_%ID,Liver_%ID,Selectivity_Index,Targeting_Efficiency,Shell Type,MW Shell,HAS_PEG,Spleen_%ID,Time_Phase_x,NP_Key,Tumor_Key,Time_Key,Time_Phase_y,Targeting_Key,Organ_Key,Administration_Key,Efficiency_Key
0,INPs-P1,Inorganic,Gold,Active,Rod,37.5,Optimal Range,-18.0,Negative,No,HeLa cell,Cervix,100.0,20.0,Tumor,6.0,1.6400,1.64,29.03,0.051589,Moderate,PEG,4441,Yes,2.76,Early,1,1,16,Early,1,17,37,3
1,INPs-P1,Inorganic,Gold,Active,Rod,37.5,Optimal Range,-18.0,Negative,No,HeLa cell,Cervix,100.0,20.0,Tumor,24.0,2.0600,2.06,31.70,0.059281,Moderate,PEG,4441,Yes,3.05,Early,1,1,23,Early,1,17,37,3
2,INPs-P1,Inorganic,Gold,Active,Rod,37.5,Optimal Range,-18.0,Negative,No,HeLa cell,Cervix,100.0,20.0,Tumor,168.0,2.0100,2.01,21.74,0.083024,Moderate,PEG,4441,Yes,2.47,Late,1,1,28,Late,1,17,37,3
3,INPs-P1,Inorganic,Gold,Active,Rod,37.5,Optimal Range,-18.0,Negative,No,HeLa cell,Cervix,100.0,20.0,Heart,6.0,0.4174,1.64,29.03,0.051589,Moderate,PEG,4441,Yes,2.76,Early,1,1,16,Early,1,4,37,3
4,INPs-P1,Inorganic,Gold,Active,Rod,37.5,Optimal Range,-18.0,Negative,No,HeLa cell,Cervix,100.0,20.0,Heart,24.0,0.3728,2.06,31.70,0.059281,Moderate,PEG,4441,Yes,3.05,Early,1,1,23,Early,1,4,37,3


### 11.2 Select Foreign Keys and Measures

Now we keep only the Foreign Keys and the numeric measures (the actual facts) and add a final Surrogate Key for the Fact table itself (`Fact_ID`).

In [ ]:
# Foreign keys + measures for the Fact table
fact_columns = [
    'NP_Key',
    'Tumor_Key',
    'Time_Key',
    'Targeting_Key',
    'Organ_Key',
    'Administration_Key',
    'Efficiency_Key',
    'Biodistribution (%ID)',
    'Tumor_%ID',
    'Liver_%ID',
    'Spleen_%ID',
    'Selectivity_Index'
]

fact_biodistribution = fact_base[fact_columns].copy()

# Add the surrogate primary key (Fact_ID)
fact_biodistribution.insert(0, 'Fact_ID', range(1, len(fact_biodistribution) + 1))

print('Fact_Biodistribution shape:', fact_biodistribution.shape)
fact_biodistribution.head()

Fact_Biodistribution shape: (2374, 13)


,Fact_ID,NP_Key,Tumor_Key,Time_Key,Targeting_Key,Organ_Key,Administration_Key,Efficiency_Key,Biodistribution (%ID),Tumor_%ID,Liver_%ID,Spleen_%ID,Selectivity_Index
0,1,1,1,16,1,17,37,3,1.6400,1.64,29.03,2.76,0.051589
1,2,1,1,23,1,17,37,3,2.0600,2.06,31.70,3.05,0.059281
2,3,1,1,28,1,17,37,3,2.0100,2.01,21.74,2.47,0.083024
3,4,1,1,16,1,4,37,3,0.4174,1.64,29.03,2.76,0.051589
4,5,1,1,23,1,4,37,3,0.3728,2.06,31.70,3.05,0.059281




## 12. Validate the Star Schema

We check three things:
1. The size of every Dimension and the Fact table.
2. That the Fact table has the same number of rows as the Silver dataset (no rows lost or duplicated during the merges).
3. That there are no missing (null) Foreign Keys, which would mean a join failed to find a match.

In [ ]:
# 1. Table sizes
print('Dim_Nanoparticle        :', dim_nanoparticle.shape)
print('Dim_Tumor               :', dim_tumor.shape)
print('Dim_Time                :', dim_time.shape)
print('Dim_Targeting_Strategy  :', dim_targeting_strategy.shape)
print('Dim_Organ               :', dim_organ.shape)
print('Dim_Administration      :', dim_administration.shape)
print('Dim_Efficiency          :', dim_efficiency.shape)
print('Fact_Biodistribution    :', fact_biodistribution.shape)
print()

# 2. Row count check: Fact table must match the Silver dataset row count
row_match = len(fact_biodistribution) == len(df)
print('Fact rows match Silver rows:', row_match)
print()

# 3. Null Foreign Key check
fk_columns = ['NP_Key', 'Tumor_Key', 'Time_Key', 'Targeting_Key', 'Organ_Key', 'Administration_Key', 'Efficiency_Key']
print('Null values in Foreign Keys:')
print(fact_biodistribution[fk_columns].isnull().sum())

Dim_Nanoparticle        : (84, 13)
Dim_Tumor               : (45, 4)
Dim_Time                : (29, 3)
Dim_Targeting_Strategy  : (2, 2)
Dim_Organ               : (17, 2)
Dim_Administration      : (42, 2)
Dim_Efficiency          : (6, 2)
Fact_Biodistribution    : (2374, 13)

Fact rows match Silver rows: True

Null values in Foreign Keys:
NP_Key                0
Tumor_Key             0
Time_Key              0
Targeting_Key         0
Organ_Key             0
Administration_Key    0
Efficiency_Key        0
dtype: int64




## 13. Export the Gold Layer Tables

Each table is saved as a separate CSV file. These eight files are the final Gold Layer deliverables, ready to be loaded into Power BI, Tableau, or a SQL Data Warehouse.

In [ ]:
import os

# Create Gold folder if it doesn't exist
gold_path = '/content/drive/MyDrive/Medallion_Arch/Gold_Layer'
os.makedirs(gold_path, exist_ok=True)

# Export Gold tables
dim_nanoparticle.to_csv(f'{gold_path}/Dim_Nanoparticle.csv', index=False)
dim_tumor.to_csv(f'{gold_path}/Dim_Tumor.csv', index=False)
dim_time.to_csv(f'{gold_path}/Dim_Time.csv', index=False)
dim_targeting_strategy.to_csv(f'{gold_path}/Dim_Targeting_Strategy.csv', index=False)
dim_organ.to_csv(f'{gold_path}/Dim_Organ.csv', index=False)
dim_administration.to_csv(f'{gold_path}/Dim_Administration.csv', index=False)
dim_efficiency.to_csv(f'{gold_path}/Dim_Efficiency.csv', index=False)
fact_biodistribution.to_csv(f'{gold_path}/Fact_Biodistribution.csv', index=False)

print(f"Saved Gold Layer to: {gold_path}")

print("\nFinal Tables:")
print(f"Dim_Nanoparticle: {dim_nanoparticle.shape[0]:,} rows | {dim_nanoparticle.shape[1]} columns")
print(f"Dim_Tumor: {dim_tumor.shape[0]:,} rows | {dim_tumor.shape[1]} columns")
print(f"Dim_Time: {dim_time.shape[0]:,} rows | {dim_time.shape[1]} columns")
print(f"Dim_Targeting_Strategy: {dim_targeting_strategy.shape[0]:,} rows | {dim_targeting_strategy.shape[1]} columns")
print(f"Dim_Organ: {dim_organ.shape[0]:,} rows | {dim_organ.shape[1]} columns")
print(f"Dim_Administration: {dim_administration.shape[0]:,} rows | {dim_administration.shape[1]} columns")
print(f"Dim_Efficiency: {dim_efficiency.shape[0]:,} rows | {dim_efficiency.shape[1]} columns")
print(f"Fact_Biodistribution: {fact_biodistribution.shape[0]:,} rows | {fact_biodistribution.shape[1]} columns")

Saved Gold Layer to: /content/drive/MyDrive/Medallion_Arch/Gold_Layer

Final Tables:
Dim_Nanoparticle: 84 rows | 13 columns
Dim_Tumor: 45 rows | 4 columns
Dim_Time: 29 rows | 3 columns
Dim_Targeting_Strategy: 2 rows | 2 columns
Dim_Organ: 17 rows | 2 columns
Dim_Administration: 42 rows | 2 columns
Dim_Efficiency: 6 rows | 2 columns
Fact_Biodistribution: 2,374 rows | 13 columns




## 14. Summary

| Step | Output | Status |
|---|---|---|
| Load Silver dataset | df (2374 x 25) | Done |
| Design Star Schema | 1 Fact + 7 Dimensions | Done |
| Dim_Nanoparticle | Unique nanoparticle records + NP_Key | Done |
| Dim_Tumor | Unique tumor records + Tumor_Key | Done |
| Dim_Time | Unique time points + Time_Key | Done |
| Dim_Targeting_Strategy | Unique targeting strategies + Targeting_Key | Done |
| Dim_Organ | Unique organs/tissues + Organ_Key | Done |
| Dim_Administration | Unique dosages + Administration_Key | Done |
| Dim_Efficiency | Ordinal Targeting Efficiency ratings + Efficiency_Key | Done |
| Fact_Biodistribution | All measures + Foreign Keys + Fact_ID | Done |
| Validation | Row counts and null checks passed | Done |
| Export | 8 CSV files saved | Done |


